In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('G036_Dataset_12_2024.csv')

In [ ]:
df_month1 = df[df['elapsed_months'] == 1].copy()
df_sorted = df_month1.sort_values(by='total_cac', ascending=False)

In [ ]:
group_size = 5
n_groups = 4 
alpha_estimates = []

In [ ]:
for group in range(n_groups):
    group_df = df_sorted.iloc[group * group_size : (group + 1) * group_size]
    
    if group_df.empty or len(group_df) < 2:
        continue

    group_df = group_df.sort_values(by='total_cac', ascending=True)
    

    baseline = group_df.iloc[0]
    baseline_total_cac = baseline['total_cac']
    baseline_ltv_cac = baseline['transaction_period_ltv_cac']
    
    alphas = []  
    

    for idx, row in group_df.iterrows():
        if row['total_cac'] == baseline_total_cac:
            continue  
        # spend_diff = row['total_cac'] - baseline_total_cac
        spend_pct = (row['total_cac'] - baseline_total_cac) / baseline_total_cac

        
        # alpha_val = ((baseline_ltv_cac / row['transaction_period_ltv_cac']) - 1) / spend_diff
        alpha_val = ((baseline_ltv_cac / row['transaction_period_ltv_cac']) - 1) / spend_pct

        alphas.append(alpha_val)
    
    if alphas:
        group_alpha = np.mean(alphas)
        alpha_estimates.append(group_alpha)
        print(f"Group {group + 1} estimated α: {group_alpha}")


In [ ]:
if alpha_estimates:
    overall_alpha = np.mean(alpha_estimates)
    print("\nOverall estimated α:", overall_alpha)
else:
    print("Not enough data to compute α.")

# Beta Calculation

In [ ]:
marketing_channels = [col for col in df_month1.columns if col.startswith('MC_')]

In [ ]:
beta_estimates = {}
baseline_spends = {}

In [ ]:
for channel in marketing_channels:
    print(f"\nChannel: {channel}")
    channel_beta_list = [] 
    baseline_list = []     
    
    df_channel_sorted = df_month1.sort_values(by=channel, ascending=False)
    
    # Process each group of top cohorts
    for group in range(n_groups):
        group_df = df_channel_sorted.iloc[group * group_size : (group + 1) * group_size]
        if group_df.empty or len(group_df) < 2:
            continue
        
        # Within the group, sort in ascending order based on channel spend to choose a baseline
        group_df = group_df.sort_values(by=channel, ascending=True)
        baseline_row = group_df.iloc[0]
        baseline_spend = baseline_row[channel]
        baseline_ltv_cac = baseline_row['transaction_period_ltv_cac']
        
        # Skip group if the baseline spend is zero (to avoid division errors)
        if baseline_spend == 0:
            continue
        
        group_betas = []
        for idx, row in group_df.iterrows():
            if row[channel] == baseline_spend:
                continue  # Skip the baseline row
            spend_increase_pct = (row[channel] - baseline_spend) / baseline_spend
            if spend_increase_pct == 0:
                continue  # Avoid division by zero
            beta_val = ((baseline_ltv_cac / row['transaction_period_ltv_cac']) - 1) / spend_increase_pct
            group_betas.append(beta_val)
        
        if group_betas:
            group_beta = np.mean(group_betas)
            channel_beta_list.append(group_beta)
            baseline_list.append(baseline_spend)
            print(f"  Group {group + 1} β: {group_beta}")
    
    if channel_beta_list:
        overall_channel_beta = np.mean(channel_beta_list)
        beta_estimates[channel] = overall_channel_beta
        baseline_spends[channel] = np.mean(baseline_list) if baseline_list else None
        print(f"Overall β for {channel}: {overall_channel_beta}")
    else:
        print(f"Not enough data to compute β for {channel}.")

In [ ]:
weights = {}
for channel in marketing_channels:
    if channel in df_month1.columns:
        corr = df_month1['transaction_period_ltv_cac'].corr(df_month1[channel])
        weights[channel] = corr
        print(f"Correlation weight for {channel}: {corr}")

In [ ]:
total_weight = sum(weights.values())
if total_weight != 0:
    normalized_weights = {channel: weight / total_weight for channel, weight in weights.items()}
else:
    normalized_weights = {channel: 0 for channel in weights}

print("\nNormalized channel weights:")
for channel, weight in normalized_weights.items():
    print(f"  {channel}: {weight}")

In [ ]:
final_beta = sum(normalized_weights.get(channel, 0) * beta_estimates.get(channel, 0)
                 for channel in marketing_channels)
print("\nFinal overall base β:", final_beta)

now whenver we calcualte new betas as marketing spends changed we need to scale the base beta to 1 so see a relative ratio

# running an experiemnt just based on alpha (considering no marketing features)

In [ ]:
top_cohort = df_month1.loc[df_month1['total_cac'].idxmax()]

In [ ]:
base_roas = top_cohort['transaction_period_ltv_cac']
print("Base ROAS from highest total_cac cohort:", base_roas)

In [ ]:
spend_increases = [0.05,0.10,0.15,0.20,0.25, 0.50, 0.75, 1.00,2.0,3,5,10]

print("\nNew ROAS estimates with increased spend:")
for inc in spend_increases:
    new_roas = base_roas * (1 / (1 + overall_alpha * inc))
    print(f"Spend increase {inc*100:.0f}%: New ROAS = {new_roas}")

# below experiment is on cohort with maximum spends but with atleast 10 months of data to see , howw will extrapolation looks after 10 months

In [ ]:
candidate_cohorts = []
for cohort in df_month1['cohort'].unique():
    cohort_data = df[df['cohort'] == cohort]
    if cohort_data['elapsed_months'].nunique() >= 10:
        candidate_cohorts.append(cohort)

if not candidate_cohorts:
    raise ValueError("No cohort has at least 10 months of data.")

In [ ]:
df_candidates = df_month1[df_month1['cohort'].isin(candidate_cohorts)]
chosen_cohort = df_candidates.sort_values(by='transaction_period_ltv_cac', ascending=False).iloc[0]['cohort']
print(f"Selected cohort: {chosen_cohort}")

cohort_data = df[(df['cohort'] == chosen_cohort) & (df['elapsed_months'] <= 10)].copy()
cohort_data = cohort_data.sort_values(by='elapsed_months')

In [ ]:
monthly_results = []
for month in range(1, 11):
    month_row = cohort_data[cohort_data['elapsed_months'] == month]
    if month_row.empty:
        continue
    observed_roas = month_row.iloc[0]['transaction_period_ltv_cac']
    
    scenario_results = {}
    for inc in spend_increases:
        predicted_roas = observed_roas / (1 + overall_alpha * inc)
        scenario_results[f"{int(inc*100)}%_Spend_Increase"] = predicted_roas
    
    monthly_results.append({
        "Elapsed Month": month,
        "ROAS_base": observed_roas,
        **scenario_results
    })

monthly_df = pd.DataFrame(monthly_results)
print("\nMonthly ROAS Table:")
print(monthly_df)

# running an experiemnt based on alpha and beta (considering no marketing features)

# we are inccreasing total spends as usual but also randomly varying the markeing spends by +- 20% to observe variation with beta

In [ ]:
base_roas = top_cohort['transaction_period_ltv_cac']
base_total_cac = top_cohort['total_cac']

In [ ]:
marketing_channels = [col for col in df_month1.columns if col.startswith('MC_')]

In [ ]:
base_channel_spends = {
    channel: top_cohort[channel]
    for channel in marketing_channels
    if pd.notnull(top_cohort[channel]) and top_cohort[channel] != 0
}

In [ ]:
base_total_marketing = sum(base_channel_spends.values())
base_ratios = {channel: spend / base_total_marketing for channel, spend in base_channel_spends.items()}

In [ ]:
print("Baseline marketing channel spends:")
print(base_channel_spends)
print("\nBaseline channel ratios:")
print(base_ratios)

In [ ]:
spend_increases = [0.05,0.10,0.15,0.20,0.25, 0.50, 0.75, 1.00,2.0,3,5,10]

In [ ]:
print("\nExperiment Results (with beta and changing channel ratios):")
for inc in spend_increases:
    new_total_cac = base_total_cac * (1 + inc)
    
    target_marketing_spend = base_total_marketing * (1 + inc)
    
    new_channel_spends_raw = {}
    for channel, spend in base_channel_spends.items():
        random_factor = np.random.uniform(0.8, 1.2)  
        new_channel_spends_raw[channel] = spend * random_factor
    
    raw_sum = sum(new_channel_spends_raw.values())
    new_channel_spends = {
        channel: spend * (target_marketing_spend / raw_sum)
        for channel, spend in new_channel_spends_raw.items()
    }
    
    new_total_marketing = sum(new_channel_spends.values()) 
    new_ratios = {channel: spend / new_total_marketing for channel, spend in new_channel_spends.items()}
    
    new_overall_beta = sum(new_ratios[channel] * beta_estimates.get(channel, 0) for channel in new_ratios)
    
    beta_decay = new_overall_beta / final_beta if final_beta != 0 else np.nan
    
    new_roas = base_roas * (1 / (1 + overall_alpha * beta_decay * inc))
    
    print(f"\n--- For total_cac increase of {inc*100:.0f}% ---")
    print("New channel spends:", new_channel_spends)
    # print("New channel ratios:", new_ratios)
    # print("New overall beta:", new_overall_beta)
    print("Beta decay (new beta / original beta):", beta_decay)
    print("New ROAS:", new_roas)

# the same 10 months expereiment but with both alpha and beta

In [ ]:
candidate_cohorts = []
for cohort in df_month1['cohort'].unique():
    cohort_data_candidate = df[df['cohort'] == cohort]
    if cohort_data_candidate['elapsed_months'].nunique() >= 10:
        candidate_cohorts.append(cohort)
if not candidate_cohorts:
    raise ValueError("No cohort with at least 10 months of data found.")

In [ ]:
df_candidates = df_month1[df_month1['cohort'].isin(candidate_cohorts)]
chosen_cohort = df_candidates.sort_values(by='transaction_period_ltv_cac', ascending=False).iloc[0]['cohort']
print(f"Selected cohort: {chosen_cohort}")

cohort_data = df[(df['cohort'] == chosen_cohort) & (df['elapsed_months'] <= 10)].copy()
cohort_data = cohort_data.sort_values(by='elapsed_months')


In [ ]:
marketing_channels = [col for col in df.columns if col.startswith("MC_")]
beta_experiment_results = []

for month in range(1, 11):
    month_data = cohort_data[cohort_data['elapsed_months'] == month]
    if month_data.empty:
        continue
    row = month_data.iloc[0]
    observed_roas = row['transaction_period_ltv_cac']
    
    baseline_channel_spends = {}
    for channel in marketing_channels:
        val = row[channel]
        if pd.notnull(val) and val != 0:
            baseline_channel_spends[channel] = val
    if not baseline_channel_spends:
        continue  
    
    baseline_total_marketing = sum(baseline_channel_spends.values())
    baseline_ratios = {channel: spend / baseline_total_marketing 
                       for channel, spend in baseline_channel_spends.items()}
    
    baseline_overall_beta = sum(baseline_ratios.get(ch, 0) * beta_estimates.get(ch, 0)
                                for ch in baseline_ratios)
    
    month_result = {"Elapsed Month": month, "ROAS_base": observed_roas}
    
    for inc in spend_increases:
        target_marketing_spend = baseline_total_marketing * (1 + inc)
        
        new_spends_raw = {}
        for channel, spend in baseline_channel_spends.items():
            random_factor = np.random.uniform(0.8, 1.2)
            new_spends_raw[channel] = spend * random_factor
        raw_total = sum(new_spends_raw.values())
        new_channel_spends = {channel: spend * (target_marketing_spend / raw_total)
                              for channel, spend in new_spends_raw.items()}
        
        new_ratios = {channel: new_channel_spends[channel] / target_marketing_spend 
                      for channel in new_channel_spends}
        
        new_overall_beta = sum(new_ratios.get(ch, 0) * beta_estimates.get(ch, 0)
                               for ch in new_ratios)
        
        beta_decay = new_overall_beta / baseline_overall_beta if baseline_overall_beta != 0 else np.nan
        
        # print(f"Month {month}, Spend Increase {int(inc*100)}%:")
        # print(f"  Baseline Overall Beta = {baseline_overall_beta:.3f}")
        # print(f"  New Overall Beta = {new_overall_beta:.3f}")
        # print(f"  Beta Decay = {beta_decay:.3f}")
        
        predicted_new_roas = observed_roas / (1 + overall_alpha * beta_decay * inc)
        
        scenario_key = f"{int(inc * 100)}%_Spend_Increase"
        month_result[scenario_key] = predicted_new_roas
    
    beta_experiment_results.append(month_result)

beta_experiment_df = pd.DataFrame(beta_experiment_results)

print("\nMonthly ROAS Predictions with Beta Adjustment:")
print(beta_experiment_df)